# Predicción de Alzas — `alzas2.ipynb`

**Objetivo:** predecir cuándo Gerardo debe poner o retirar alzas (extensiones de colmena para la miel) en el apiario, usando los datos de sensores de 17 colmenas de 2023 a 2026.

**Pipeline:**
1. Carga y limpieza de datos brutos (sensores IoT, 15-min → diario)
2. Registro manual de intervenciones de alzas por año
3. Ingeniería de features: peso, acústica, temperatura, perfil histórico, comparativa entre colmenas
4. Modelos XGBoost con distintas configuraciones de features y ventanas de predicción
5. Validación temporal (train 2023-2025 → test 2026) y walk-forward CV
6. **Modelo final:** sin features temporales, ventana 14 días (AUC 0.727, AP 0.371)

**Métrica principal:** Average Precision (AP) — más informativa que AUC para datos muy desbalanceados (~5% positivos).


## 1. Imports y carga de datos

In [47]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

In [48]:
df_raw = pd.read_csv('../../data/12062026all_boxes.csv')  # actualizado el 12/06/2026 con datos de 2023-2026

df_raw['Time'] = pd.to_datetime(df_raw['Time'])
df_raw = df_raw.rename(columns={'Hive name': 'box_id'})
df_raw = df_raw.sort_values(['box_id', 'Time']).reset_index(drop=True)

print(f"Dataset cargado: {df_raw.shape[0]:,} registros")
print(f"Colmenas: {sorted(df_raw['box_id'].unique())}")
print(f"Período: {df_raw['Time'].min()} → {df_raw['Time'].max()}")
print(f"\nColumnas: {list(df_raw.columns)}")
print(f"\nMissing values (%):")
print((df_raw.isnull().sum() / len(df_raw) * 100).round(1))

Dataset cargado: 4,556,275 registros
Colmenas: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]
Período: 2023-01-01 00:00:21 → 2026-06-16 16:24:56

Columnas: ['box_id', 'Time', 'Temperature heart', 'Humidity heart', 'Frequency', 'Volume', 'Temperature scale', 'Humidity scale', 'Weight']

Missing values (%):
box_id                0.0
Time                  0.0
Temperature heart    59.7
Humidity heart       59.7
Frequency            59.7
Volume               59.7
Temperature scale    44.1
Humidity scale       44.1
Weight               44.1
dtype: float64


## 2. Registro de intervenciones de alzas (2023–2026)

Cosechas de miel excluidas del modelo — solo se etiquetan ADD/REMOVE de alzas.

In [49]:
# ============================================================
# ALZAS (2025 + 2026)
# ============================================================
# Estados especiales por colmena y período
estados_especiales_2025 = {
    # (box_id, fecha_inicio, fecha_fin): estado
    (10, '2025-01-01', '2025-12-31'): 'vacia',   # vacia todo 2025
    (11, '2025-01-01', '2025-12-31'): 'vacia',
    (12, '2025-01-01', '2025-12-31'): 'vacia',
    (16, '2025-01-01', '2025-12-31'): 'vacia',
    (17, '2025-01-01', '2025-12-31'): 'vacia',
    (18, '2025-01-01', '2025-12-31'): 'vacia',
    # Colmenas con eventos especiales
    (1,  '2025-03-08', '2025-12-31'): 'baja',    # Baja en marzo
    (5,  '2025-03-08', '2025-12-31'): 'baja',
    (4,  '2025-02-26', '2025-12-31'): 'partida', # Dividida en febrero
    (10, '2025-03-08', '2025-12-31'): 'nueva',   # Nueva en marzo
}

alzas_2025 = {
    '2025-01-23': {13: 1},
    '2025-02-26': {2:1, 3:1, 13:1, 15:1},
    '2025-03-08': {4:1},                          # box1=Baja, box5=Baja ignorados
    '2025-03-17': {3:1, 4:1, 6:1, 14:1},          # box1=vacia ignorado
    '2025-03-25': {7:1, 9:1, 10:1, 15:1},
    '2025-03-31': {15:1},
    '2025-04-07': {6:1, 10:-1, 13:-1, 15:1},
    '2025-04-13': {2:1, 6:1, 7:-1, 8:1, 13:1, 14:1},
    '2025-04-23': {3:1, 6:1, 8:1, 13:1, 14:1, 15:1},
}

# Cosecha de miel — intervención manual de peso
cosecha_miel = {
    'fecha': pd.to_datetime('2025-05-12'),
    'descripcion': 'Cosecha de miel — bajada manual de peso en todas las colmenas'
}

alzas_2026 = {
    '2026-02-27': {1:1, 17:1},
    '2026-03-13': {1:1, 11:1, 17:1},
    '2026-03-26': {9:1},
    '2026-04-08': {4:1, 5:1, 9:1, 11:1},
    '2026-05-07': {3:1, 8:1, 9:1, 14:1, 16:1},
    '2026-06-09': {7:1, 12:1, 13:1, 15:1, 16:1},
}

# Colmenas activas en 2025 (con sensores Y no vacías)
colmenas_activas_2025 = [1, 2, 3, 4, 5, 6, 7, 8, 9, 13, 14, 15]
colmenas_activas_2026 = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17]


In [50]:
# ============================================================
# ALZAS (2023 + 2024)
# ============================================================
alzas_2023 = {
    '2023-03-20': {1:1, 5:1, 8:1, 9:1, 10:1},
    '2023-04-13': {3:1, 4:1},
    '2023-04-29': {3:1},   # 2ª alza box3
    '2023-05-04': {6:1},
    '2023-06-02': {3:1, 4:1, 5:1, 8:1},  # 2ª y 3ª alzas
    # 2023-07-05: Cosecha de miel — NO es intervención de alza
}

alzas_2024 = {
    '2024-03-28': {1:3, 5:1, 12:2},  # múltiples alzas de golpe
    '2024-04-06': {3:1},
    '2024-04-08': {8:1},
    '2024-04-12': {1:1, 3:1, 5:1, 8:1, 11:1},
    # Gerardo tiene más notas de 2024 pendientes de buscar
}

## 3. Dataset de alzas (2023–2026)

In [51]:
# Unificar todo
alzas_todas = {**alzas_2023, **alzas_2024,
               **alzas_2025, **alzas_2026}

registros_todos = []
for fecha, colmenas in alzas_todas.items():
    for box_id, delta in colmenas.items(): #delta = +1 alza, -1 bajada, 2 o 3 para múltiples alzas
        # Para 2024-03-28 con múltiples alzas, contar como 1 intervención
        registros_todos.append({
            'fecha':      pd.to_datetime(fecha),
            'box_id':     int(box_id),
            'delta_alza': delta,
            'accion':     'ADD' if delta > 0 else 'REMOVE',
            'year':       pd.to_datetime(fecha).year,
        })

df_alzas_final = pd.DataFrame(registros_todos).sort_values(
    ['box_id','fecha']).reset_index(drop=True)

print("=== DATASET COMPLETO DE ALZAS (2023-2026) ===\n")
print(f"Total intervenciones: {len(df_alzas_final)}")
print(f"Colmenas con alzas: {sorted(df_alzas_final['box_id'].unique())}")
print(f"Período: {df_alzas_final['fecha'].min().date()} → "
      f"{df_alzas_final['fecha'].max().date()}")

print(f"\nPor año:")
by_year = df_alzas_final.groupby(['year','accion']).size().unstack(fill_value=0)
print(by_year)

print(f"\nTotal intervenciones por año:")
print(df_alzas_final.groupby('year').size())

print(f"\nNota de Gerardo:")
print(f"  - Alzas principalmente en primavera (marzo-junio)")
print(f"  - Cosechas: 05/07/2023, 12/05/2025 (excluidas del modelo)")
print(f"  - Velutina: desde su llegada evitan poner alzas")
print(f"  - Pendiente: más notas de 2024 de Gerardo")

=== DATASET COMPLETO DE ALZAS (2023-2026) ===

Total intervenciones: 74
Colmenas con alzas: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]
Período: 2023-03-20 → 2026-06-09

Por año:
accion  ADD  REMOVE
year               
2023     13       0
2024     10       0
2025     28       3
2026     20       0

Total intervenciones por año:
year
2023    13
2024    10
2025    31
2026    20
dtype: int64

Nota de Gerardo:
  - Alzas principalmente en primavera (marzo-junio)
  - Cosechas: 05/07/2023, 12/05/2025 (excluidas del modelo)
  - Velutina: desde su llegada evitan poner alzas
  - Pendiente: más notas de 2024 de Gerardo


## 4. Features diarias

In [52]:
def build_daily_features(df_raw):
    
    df_scale = df_raw.dropna(subset=['Weight']).copy()
    df_heart = df_raw.dropna(subset=['Frequency']).copy()

    # Filtro 1: rango absoluto físicamente posible
    df_scale = df_scale[
        (df_scale["Weight"] >= 10) &
        (df_scale["Weight"] <= 120)
    ]

    # Filtro 2: IQR por colmena con factor alto (5×)
    # Solo para eliminar valores individuales extremos
    # NO elimina períodos completos
    df_scale_clean = []
    for box_id, group in df_scale.groupby('box_id'):
        Q1  = group["Weight"].quantile(0.25)
        Q3  = group["Weight"].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 5 * IQR
        upper = Q3 + 5 * IQR
        filtered = group[
            (group["Weight"] >= lower) &
            (group["Weight"] <= upper)
        ]
        n_removed = len(group) - len(filtered)
        if n_removed > 0:
            print(f"  Box{box_id}: {n_removed} registros eliminados "
                  f"(fuera de [{lower:.1f}, {upper:.1f}] kg)")
        df_scale_clean.append(filtered)
    df_scale = pd.concat(df_scale_clean)

    # Agregar diariamente
    daily_scale = df_scale.groupby(
        ['box_id', df_scale['Time'].dt.date]
    ).agg(
        Weight         = ('Weight',            'mean'),
        Weight_max     = ('Weight',            'max'),
        Weight_min     = ('Weight',            'min'),
        Temp_scale     = ('Temperature scale', 'mean'),
        Humidity_scale = ('Humidity scale',    'mean'),
    ).reset_index().rename(columns={'Time': 'date'})
    
    daily_heart = df_heart.groupby(
        ['box_id', df_heart['Time'].dt.date]
    ).agg(
        Frequency      = ('Frequency',         'mean'),
        Freq_std       = ('Frequency',         'std'),
        Temp_heart     = ('Temperature heart', 'mean'),
        Humidity_heart = ('Humidity heart',    'mean'),
        Volume         = ('Volume',            'mean'),
    ).reset_index().rename(columns={'Time': 'date'})
    
    daily = daily_scale.merge(daily_heart, on=['box_id','date'], how='left')
    daily['date']   = pd.to_datetime(daily['date'])
    daily = daily.sort_values(['box_id','date']).reset_index(drop=True)
    
    # Features temporales
    daily['month']     = daily['date'].dt.month
    daily['dayofyear'] = daily['date'].dt.dayofyear
    daily['season']    = daily['month'].map({
        12:0, 1:0, 2:0,
        3:1,  4:1, 5:1,
        6:2,  7:2, 8:2,
        9:3,  10:3,11:3
    })
    
    # Features de peso
    grp = daily.groupby('box_id')['Weight']
    daily['weight_diff_1d']   = grp.diff(1)
    daily['weight_diff_7d']   = grp.diff(7)
    daily['weight_diff_14d']  = grp.diff(14)
    daily['weight_ma_7d']     = grp.transform(
        lambda x: x.rolling(7, min_periods=3).mean())
    daily['weight_ma_14d']    = grp.transform(
        lambda x: x.rolling(14, min_periods=5).mean())
    daily['weight_std_7d']    = grp.transform(
        lambda x: x.rolling(7, min_periods=3).std())
    daily['weight_amplitude'] = daily['Weight_max'] - daily['Weight_min']
    
    # Correlación rolling Weight↔Temp
    daily['corr_w_temp'] = (
        daily.groupby('box_id', group_keys=False)
        .apply(lambda df: df['Weight'].rolling(14, min_periods=7)
               .corr(df['Temp_scale']))
    )
    
    # Features acústicas
    daily['freq_ma_7d'] = (
        daily.groupby('box_id')['Frequency']
        .transform(lambda x: x.rolling(7, min_periods=3).mean())
    )
    daily['freq_diff_7d'] = daily.groupby('box_id')['Frequency'].diff(7)
    
    # Filtro 3: eliminar períodos conocidos con sensor inestable
    # Identificados manualmente tras análisis exhaustivo
    periodos_inestables = [
        # (box_id, fecha_inicio, fecha_fin, motivo)
        (9,  '2023-08-15', '2023-09-30', 'sensor inestable'),
        (11, '2026-02-01', '2026-04-15', 'sensor inestable'),
        (17, '2025-06-01', '2025-07-31', 'sensor inestable'),
        (8,  '2024-09-20', '2024-10-15', 'sensor inestable'),
        (8,  '2024-12-15', '2025-03-10', 'sensor inestable'),
        (17, '2026-01-07', '2026-01-07', 'gap transition'),
        (6,  '2024-05-10', '2024-05-10', 'sensor dropout'),
        (8,  '2024-11-27', '2024-12-05', 'sensor inestable residual'),
    ]
    
    mask_inestable = pd.Series(False, index=daily.index)
    for box_id, inicio, fin, motivo in periodos_inestables:
        mask = (
            (daily['box_id'] == box_id) &
            (daily['date'] >= inicio) &
            (daily['date'] <= fin)
        )
        n = mask.sum()
        mask_inestable |= mask
        print(f"  Box{box_id} {inicio}→{fin} ({motivo}): {n} días")
    
    daily = daily[~mask_inestable]
    
    # Filtro 4: eliminar días de transición tras gaps largos
    # (el primer día tras >30 días sin datos tiene diff imposible)
    daily = daily.sort_values(['box_id','date'])
    daily['days_gap'] = daily.groupby('box_id')['date'].diff().dt.days
    mask_gap = daily['days_gap'] > 30
    n_gap = mask_gap.sum()
    if n_gap > 0:
        print(f"\n  Días tras gap >30 días (diff no fiable): {n_gap}")
        daily.loc[mask_gap, 'weight_diff_1d'] = np.nan
    daily = daily.drop(columns=['days_gap'])
    
    # Streak (tras limpiar diffs de gaps)
    def calc_streak(series):
        growing = (series > 0).astype(int)
        streak  = growing * (growing.groupby(
            (growing != growing.shift()).cumsum()).cumcount() + 1)
        return streak
    
    daily['weight_growing_streak'] = (
        daily.groupby('box_id')['weight_diff_1d']
        .transform(calc_streak)
    )
    
    # Aceleración del peso (segunda derivada)
    daily['weight_acceleration'] = daily.groupby('box_id')['weight_diff_1d'].diff(1)

    # Encoding cíclico día del año (mejor que raw dayofyear en árboles)
    daily['sin_dayofyear'] = np.sin(2 * np.pi * daily['dayofyear'] / 365)
    daily['cos_dayofyear'] = np.cos(2 * np.pi * daily['dayofyear'] / 365)

    # Días dentro de la temporada activa (0 = 1 marzo, crece hasta finales junio)
    spring_start = pd.to_datetime({
        'year': daily['date'].dt.year, 'month': 3, 'day': 1})
    daily['days_in_season'] = (daily['date'] - spring_start).dt.days.clip(lower=0)

    # Tendencia peso a 21 días
    daily['weight_diff_21d'] = daily.groupby('box_id')['Weight'].diff(21)

    # Slope lineal del peso (14 días) — más robusto que diff en datos ruidosos
    def _slope(arr):
        valid = arr[~np.isnan(arr)]
        if len(valid) < 5:
            return np.nan
        return float(np.polyfit(np.arange(len(valid)), valid, 1)[0])

    daily['weight_trend_slope'] = (
        daily.groupby('box_id')['Weight']
        .transform(lambda x: x.rolling(14, min_periods=5).apply(_slope, raw=True))
    )

    # Días con ganancia positiva en últimos 7 días — racha consistente
    daily['n_positive_days_7d'] = (
        daily.groupby('box_id')['weight_diff_1d']
        .transform(lambda x: (x > 0).rolling(7, min_periods=3).sum())
    )

    # Tendencia de temperatura exterior (llegada del buen tiempo anticipa floración)
    daily['temp_trend_7d'] = daily.groupby('box_id')['Temp_scale'].diff(7)

    return daily

print("Construyendo features diarias...")
print("\nIQR eliminaciones:")
daily = build_daily_features(df_raw)
print(f"\nDataset diario: {daily.shape[0]:,} filas | {daily.shape[1]} columnas")
print(f"Período: {daily['date'].min().date()} → {daily['date'].max().date()}")
print(f"\nPesos máximos por colmena:")
print(daily.groupby('box_id')['Weight'].agg(
    ['min','max','mean','count']).round(1).to_string())
print(f"\nMax |weight_diff_1d| (excl. NaN): "
      f"{daily['weight_diff_1d'].abs().max():.1f} kg")
print(f"\nMissing (%):")
print((daily.isnull().sum() / len(daily) * 100).round(1)
      [lambda x: x > 0].to_string())

Construyendo features diarias...

IQR eliminaciones:
  Box11: 1615 registros eliminados (fuera de [-35.6, 93.9] kg)
  Box9 2023-08-15→2023-09-30 (sensor inestable): 12 días
  Box11 2026-02-01→2026-04-15 (sensor inestable): 54 días
  Box17 2025-06-01→2025-07-31 (sensor inestable): 23 días
  Box8 2024-09-20→2024-10-15 (sensor inestable): 14 días
  Box8 2024-12-15→2025-03-10 (sensor inestable): 38 días
  Box17 2026-01-07→2026-01-07 (gap transition): 1 días
  Box6 2024-05-10→2024-05-10 (sensor dropout): 1 días
  Box8 2024-11-27→2024-12-05 (sensor inestable residual): 8 días

  Días tras gap >30 días (diff no fiable): 22

Dataset diario: 15,249 filas | 34 columnas
Período: 2023-01-01 → 2026-06-16

Pesos máximos por colmena:
         min    max  mean  count
box_id                          
1       10.1  106.7  41.6   1029
2       12.1   73.3  34.3   1209
3       13.6  115.7  41.7   1171
4       20.0   78.7  39.6   1242
5       10.1   70.4  31.6   1248
6       10.0   73.8  29.3    806
7      

In [53]:
#save dataset
daily.to_csv('daily_data.csv', index=False)

## 5. Investigación de pesos extremos

Algunas colmenas (Box1, Box9, Box15, Box17) alcanzan pesos > 80 kg debido al alzas acumuladas.
Se verifican los saltos bruscos para distinguir errores de sensor de crecimientos reales.


In [54]:
# Investigar los pesos extremos

print("=== INVESTIGACIÓN DE PESOS EXTREMOS ===\n")

# Colmenas con pesos > 100 kg
for box_id in [1, 9, 15, 17]:
    datos_extremos = daily[
        (daily['box_id'] == box_id) & 
        (daily['Weight'] > 80)
    ][['date','Weight','weight_diff_1d']].sort_values('date')
    
    if len(datos_extremos) == 0:
        continue
    
    print(f"Box{box_id} — días con Weight > 80 kg:")
    print(f"  Max: {daily[daily['box_id']==box_id]['Weight'].max():.1f} kg")
    print(f"  Período: {datos_extremos['date'].min().date()} → "
          f"{datos_extremos['date'].max().date()}")
    
    # Ver si hay saltos bruscos (posible error de sensor)
    saltos = datos_extremos[datos_extremos['weight_diff_1d'].abs() > 10]
    if len(saltos) > 0:
        print(f"  ⚠ Saltos > 10 kg en un día:")
        print(saltos.to_string(index=False))
    else:
        print(f"  ✓ Sin saltos bruscos — crecimiento gradual")
    print()

# Específicamente Box17 que llega a 145 kg
print("Box17 — evolución completa de peso:")
box17 = daily[daily['box_id']==17][['date','Weight','weight_diff_1d']].sort_values('date')
print(box17.to_string(index=False))

=== INVESTIGACIÓN DE PESOS EXTREMOS ===

Box1 — días con Weight > 80 kg:
  Max: 106.7 kg
  Período: 2024-04-09 → 2024-05-29
  ⚠ Saltos > 10 kg en un día:
      date    Weight  weight_diff_1d
2024-05-07 80.404028      -10.053333

Box9 — días con Weight > 80 kg:
  Max: 90.0 kg
  Período: 2025-06-02 → 2026-05-17
  ✓ Sin saltos bruscos — crecimiento gradual

Box15 — días con Weight > 80 kg:
  Max: 119.9 kg
  Período: 2025-04-28 → 2025-07-03
  ✓ Sin saltos bruscos — crecimiento gradual

Box17 — evolución completa de peso:
      date    Weight  weight_diff_1d
2026-01-08 34.711458        0.026843
2026-01-09 33.907500       -0.803958
2026-01-10 34.069028        0.161528
2026-01-11 34.254167        0.185139
2026-01-12 34.042014       -0.212153
2026-01-13 33.864653       -0.177361
2026-01-14 33.811181       -0.053472
2026-01-15 33.590069       -0.221111
2026-01-16 33.442500       -0.147569
2026-01-17 33.923542        0.481042
2026-01-18 34.542083        0.618542
2026-01-19 34.625694        0.083

## 6. Etiquetado del target

**Target:** 1 si en los próximos `N` días hay una intervención de alza, 0 en caso contrario.
La función `etiquetar_target` crea también la columna `dias_hasta_intervencion` para análisis exploratorio.


In [55]:
# ============================================================
# ETIQUETADO — target para el modelo
# ============================================================

def etiquetar_target(daily, df_alzas, dias_pre=14, dias_post=3):
    """
    Para cada día y colmena, etiquetar si en los próximos
    dias_pre días habrá una intervención de alzas
    
    Target:
    0 = No action needed
    1 = ADD alza coming in next 14 days
    2 = REMOVE alza coming in next 14 days
    """
    daily = daily.copy()
    daily['target'] = 0
    daily['dias_hasta_intervencion'] = np.nan
    
    for _, interv in df_alzas.iterrows():
        box  = interv['box_id']
        fecha = interv['fecha']
        accion = 1 if interv['accion'] == 'ADD' else 2
        
        # Marcar los dias_pre días anteriores a la intervención
        mask = (
            (daily['box_id'] == box) &
            (daily['date'] >= fecha - pd.Timedelta(days=dias_pre)) &
            (daily['date'] <  fecha)
        )
        daily.loc[mask, 'target'] = accion
        
        # Días hasta la intervención
        for idx in daily[mask].index:
            dias = (fecha - daily.loc[idx, 'date']).days
            # Solo sobreescribir si está más cerca
            if pd.isna(daily.loc[idx, 'dias_hasta_intervencion']) or \
               dias < daily.loc[idx, 'dias_hasta_intervencion']:
                daily.loc[idx, 'dias_hasta_intervencion'] = dias
    
    print("=== DISTRIBUCIÓN DEL TARGET ===")
    counts = daily['target'].value_counts().sort_index()
    labels = {0: 'NO ACTION', 1: 'ADD alza', 2: 'REMOVE alza'}
    for k, v in counts.items():
        print(f"  {labels[k]}: {v:,} ({v/len(daily)*100:.1f}%)")
    
    # Solo colmenas con alzas etiquetadas
    boxes_con_alzas = df_alzas['box_id'].unique()
    daily_labeled = daily[daily['box_id'].isin(boxes_con_alzas)].copy()
    
    print(f"\nDataset filtrado (colmenas con alzas): {len(daily_labeled):,} filas")
    counts2 = daily_labeled['target'].value_counts().sort_index()
    for k, v in counts2.items():
        print(f"  {labels[k]}: {v:,} ({v/len(daily_labeled)*100:.1f}%)")
    
    return daily, daily_labeled

daily, daily_labeled = etiquetar_target(daily, df_alzas_final)

=== DISTRIBUCIÓN DEL TARGET ===
  NO ACTION: 14,435 (94.7%)
  ADD alza: 780 (5.1%)
  REMOVE alza: 34 (0.2%)

Dataset filtrado (colmenas con alzas): 15,115 filas
  NO ACTION: 14,301 (94.6%)
  ADD alza: 780 (5.2%)
  REMOVE alza: 34 (0.2%)


## 7. Features históricas por colmena

Permiten distinguir colmenas con distinto tamaño y ritmo de crecimiento — sin esto el modelo confunde una colmena ligera con una que necesita alza.

In [56]:
# Celda — Features relativas por colmena (perfil histórico)
# Target binario: 0=NO ACTION, 1=ACTUAR (ADD o REMOVE)
daily_labeled['target_binary'] = (daily_labeled['target'] > 0).astype(int)
daily_model = daily_labeled.copy()
# Excluir período de cosecha (12-15 mayo 2025)
daily_model_clean = daily_model[
    ~(
        (daily_model['date'] >= '2025-05-12') &
        (daily_model['date'] <= '2025-05-15')
    )
].copy()

# Excluir colmenas vacías del período correspondiente (derivado de estados_especiales_2025)
vacias_2025 = [k[0] for k, v in estados_especiales_2025.items() if v == 'vacia']
mask_vacias_2025 = (
    (daily_model_clean['date'].dt.year == 2025) &
    (daily_model_clean['box_id'].isin(vacias_2025))
)
daily_model_clean = daily_model_clean[~mask_vacias_2025]

print(f"Dataset antes: {len(daily_model):,} filas")
print(f"Dataset después de limpieza: {len(daily_model_clean):,} filas")
print(f"Filas eliminadas: {len(daily_model) - len(daily_model_clean):,}")
print(f"\nColmenas en dataset limpio: {sorted(daily_model_clean['box_id'].unique())}")

def add_hive_relative_features(daily_clean, df_alzas_clean):
    """
    Features relativas por colmena respecto a su propio histórico.
    Implementación vectorizada: ~50x más rápida que la versión con iterrows.
    """
    df = daily_clean.copy().sort_values(['box_id', 'date'])

    # Inicializar columnas
    for col in ['weight_historical_mean', 'weight_historical_std',
                'weight_vs_historical', 'weight_pct_of_max',
                'days_since_last_alza', 'days_since_last_ADD',
                'n_alzas_this_season', 'weight_growing_streak']:
        df[col] = np.nan

    for box_id in df['box_id'].unique():
        mask  = df['box_id'] == box_id
        df_box = df[mask].sort_values('date')
        idx    = df_box.index

        # ── Peso histórico por mes (expanding, solo pasado) ──────────────
        for month in df_box['date'].dt.month.unique():
            m_mask = df_box['date'].dt.month == month
            df_m   = df_box[m_mask].sort_values('date')
            if len(df_m) < 2:
                continue
            # expanding().shift(1) → estadísticas de fechas anteriores, no la actual
            h_mean = df_m['Weight'].expanding(min_periods=7).mean().shift(1)
            h_std  = df_m['Weight'].expanding(min_periods=7).std().shift(1)
            df.loc[df_m.index, 'weight_historical_mean'] = h_mean.values
            df.loc[df_m.index, 'weight_historical_std']  = h_std.values
            df.loc[df_m.index, 'weight_vs_historical'] = (
                (df_m['Weight'].values - h_mean.values) / (h_std.values + 1e-6)
            )

        # ── % del máximo histórico (expanding p95) ───────────────────────
        exp_max = df_box['Weight'].expanding(min_periods=14).quantile(0.95).shift(1)
        df.loc[idx, 'weight_pct_of_max'] = (
            df_box['Weight'].values / (exp_max.values + 1e-6)
        )

        # ── Días desde última alza (cualquier tipo) — merge_asof ─────────
        alzas_b = df_alzas_clean[
            df_alzas_clean['box_id'] == box_id].sort_values('fecha')
        alzas_b_add = alzas_b[alzas_b['accion'] == 'ADD'].sort_values('fecha')

        # date-1 → semántica 'fecha < date' (excluye la intervención del día actual)
        dates_df = df_box[['date']].sort_values('date').reset_index()
        dates_df['date_excl'] = dates_df['date'] - pd.Timedelta(days=1)

        if len(alzas_b) > 0:
            merged = pd.merge_asof(
                dates_df, alzas_b[['fecha']],
                left_on='date_excl', right_on='fecha', direction='backward'
            )
            days = (dates_df['date'] - merged['fecha']).dt.days.fillna(999)
            df.loc[dates_df['index'], 'days_since_last_alza'] = days.values
        else:
            df.loc[idx, 'days_since_last_alza'] = 999

        if len(alzas_b_add) > 0:
            merged_add = pd.merge_asof(
                dates_df, alzas_b_add[['fecha']],
                left_on='date_excl', right_on='fecha', direction='backward'
            )
            days_add = (dates_df['date'] - merged_add['fecha']).dt.days.fillna(999)
            df.loc[dates_df['index'], 'days_since_last_ADD'] = days_add.values
        else:
            df.loc[idx, 'days_since_last_ADD'] = 999

        # ── Alzas ADD acumuladas en la temporada — searchsorted ──────────
        for year in df_box['date'].dt.year.unique():
            y_mask  = mask & (df['date'].dt.year == year)
            dates_y = df.loc[y_mask, 'date'].sort_values()
            adds_y  = alzas_b_add[
                alzas_b_add['fecha'].dt.year == year
            ]['fecha'].sort_values().values
            if len(adds_y) > 0:
                n = np.searchsorted(adds_y, dates_y.values, side='left')
            else:
                n = np.zeros(len(dates_y), dtype=int)
            df.loc[dates_y.index, 'n_alzas_this_season'] = n

        # ── Racha de días consecutivos con ganancia de peso ──────────────
        ganando = (df_box['weight_diff_1d'] > 0).astype(int)
        streak  = ganando.groupby(
            (ganando != ganando.shift()).cumsum()).cumsum()
        df.loc[idx, 'weight_growing_streak'] = streak.values

    new_features = ['weight_historical_mean', 'weight_historical_std',
                    'weight_vs_historical', 'weight_pct_of_max',
                    'days_since_last_alza', 'days_since_last_ADD',
                    'n_alzas_this_season', 'weight_growing_streak']

    print('Features añadidas:')
    for f in new_features:
        pct_nan = df[f].isnull().sum() / len(df) * 100
        print(f'  {f:<30}: {pct_nan:.1f}% NaN | '
              f'mean={df[f].mean():.2f} | std={df[f].std():.2f}')

    return df, new_features

print("Calculando features históricas por colmena...")
print("(puede tardar unos minutos)\n")
daily_enhanced, new_features = add_hive_relative_features(
    daily_model_clean, df_alzas_final)

Dataset antes: 15,115 filas
Dataset después de limpieza: 14,135 filas
Filas eliminadas: 980

Colmenas en dataset limpio: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]
Calculando features históricas por colmena...
(puede tardar unos minutos)

Features añadidas:
  weight_historical_mean        : 9.5% NaN | mean=34.04 | std=12.68
  weight_historical_std         : 9.5% NaN | mean=6.77 | std=6.57
  weight_vs_historical          : 9.5% NaN | mean=7044.65 | std=404882.13
  weight_pct_of_max             : 1.7% NaN | mean=0.68 | std=0.29
  days_since_last_alza          : 0.0% NaN | mean=458.91 | std=391.85
  days_since_last_ADD           : 0.0% NaN | mean=459.64 | std=391.46
  n_alzas_this_season           : 0.0% NaN | mean=0.87 | std=1.22
  weight_growing_streak         : 0.0% NaN | mean=1.91 | std=4.46


## 8. Features de comparación entre colmenas (apiario)

Para cada día se calcula cómo está cada colmena respecto al promedio del apiario ese mismo día:
- `weight_vs_apiary`: desviación normalizada respecto a la media del apiario
- `weight_rank_pct`: percentil de peso dentro del apiario
- `weight_apiary_ratio`: ratio peso colmena / peso medio apiario


In [57]:
# Celda — Features de comparación entre colmenas + ventana 7 días

def add_apiary_comparison_features(daily_enhanced):
    """
    Features que comparan cada colmena con el resto del apiario
    en el mismo día — captura si una colmena está destacando
    respecto a sus vecinas (señal de que necesita intervención)
    """
    df = daily_enhanced.copy().sort_values(['date','box_id'])
    
    # Media y std del apiario cada día
    apiary_daily = df.groupby('date').agg(
        apiary_weight_mean = ('Weight', 'mean'),
        apiary_weight_std  = ('Weight', 'std'),
        apiary_growth_mean = ('weight_diff_7d', 'mean'),
        apiary_growth_std  = ('weight_diff_7d', 'std'),
        n_hives_active     = ('Weight', 'count'),
    ).reset_index()
    
    df = df.merge(apiary_daily, on='date', how='left')
    
    # Posición relativa de cada colmena respecto al apiario
    df['weight_vs_apiary']   = (
        (df['Weight'] - df['apiary_weight_mean']) / 
        (df['apiary_weight_std'] + 1e-6)
    )  # z-score dentro del apiario ese día
    
    df['growth_vs_apiary']   = (
        (df['weight_diff_7d'] - df['apiary_growth_mean']) /
        (df['apiary_growth_std'] + 1e-6)
    )  # crece más rápido que la media del apiario?
    
    df['weight_rank_pct']    = df.groupby('date')['Weight'].rank(pct=True)
    # percentil de peso dentro del apiario ese día (1.0 = la más pesada)
    
    df['growth_rank_pct']    = df.groupby('date')['weight_diff_7d'].rank(pct=True)
    # percentil de crecimiento dentro del apiario ese día
    
    # Ratio peso colmena / media apiario
    df['weight_apiary_ratio'] = df['Weight'] / (df['apiary_weight_mean'] + 1e-6)
    
    apiary_features = [
        'weight_vs_apiary', 'growth_vs_apiary',
        'weight_rank_pct',  'growth_rank_pct',
        'weight_apiary_ratio'
    ]
    
    print("Features de comparación entre colmenas:")
    for f in apiary_features:
        pct_nan = df[f].isnull().sum() / len(df) * 100
        print(f"  {f:<25}: {pct_nan:.1f}% NaN | "
              f"mean={df[f].mean():.2f} | std={df[f].std():.2f}")
    
    return df, apiary_features

print("Añadiendo features de comparación entre colmenas...")
daily_final, apiary_features = add_apiary_comparison_features(daily_enhanced)

Añadiendo features de comparación entre colmenas...
Features de comparación entre colmenas:
  weight_vs_apiary         : 0.0% NaN | mean=0.00 | std=0.95
  growth_vs_apiary         : 0.8% NaN | mean=-0.00 | std=0.95
  weight_rank_pct          : 0.0% NaN | mean=0.54 | std=0.29
  growth_rank_pct          : 0.8% NaN | mean=0.54 | std=0.29
  weight_apiary_ratio      : 0.0% NaN | mean=1.00 | std=0.33


## 9. Features de umbral y reglas de negocio

Features binarias basadas en reglas de Gerardo:
- `above_min_weight`: la colmena supera el peso mínimo para poner alza (≥ 20 kg en temporada)
- `overdue_for_alza`: lleva más de 200 días sin intervención en temporada activa
- `small_but_growing`: colmena pequeña pero en crecimiento (podría necesitar alza pronto)


In [58]:
# Añadir feature de umbral mínimo estacional

def add_threshold_features(df, min_weight=20, months_active=[3,4,5,6,7]):
    df = df.copy()
    
    # Colmena supera umbral mínimo en temporada activa
    df['above_min_weight'] = (
        (df['Weight'] >= min_weight) &
        (df['month'].isin(months_active))
    ).astype(int)
    
    # Lleva mucho tiempo sin intervención (>200 días) en temporada
    df['overdue_for_alza'] = (
        (df['days_since_last_alza'] > 200) &
        (df['month'].isin(months_active)) &
        (df['Weight'] >= min_weight)
    ).astype(int)
    
    # Peso relativo dentro del apiario — percentil bajo pero activa
    # (colmenas pequeñas pero en crecimiento en temporada)
    df['small_but_growing'] = (
        (df['weight_rank_pct'] < 0.5) &      # por debajo de la mediana
        (df['weight_diff_14d'] > 1.0) &       # pero creciendo
        (df['month'].isin(months_active))
    ).astype(int)
    
    threshold_features = [
        'above_min_weight', 'overdue_for_alza', 'small_but_growing'
    ]
    
    print("Features de umbral añadidas:")
    for f in threshold_features:
        print(f"  {f}: {df[f].sum()} días activos ({df[f].mean()*100:.1f}%)")
    
    return df, threshold_features

daily_threshold, threshold_features = add_threshold_features(daily_final)

Features de umbral añadidas:
  above_min_weight: 5757 días activos (40.7%)
  overdue_for_alza: 2429 días activos (17.2%)
  small_but_growing: 1035 días activos (7.3%)


## 10. Filtro estacional y features del modelo

In [59]:
daily_threshold = daily_threshold[daily_threshold['month'].between(2, 6)]

daily_threshold.to_csv('daily_features_final.csv', index=False)

feature_cols = [
    # Peso y derivadas
    'Weight', 'weight_diff_1d', 'weight_diff_7d', 'weight_diff_14d',
    'weight_ma_7d', 'weight_ma_14d', 'weight_std_7d', 'weight_amplitude',
    # Correlación — nuestro hallazgo original
    'corr_w_temp',
    # Temperatura y humedad exterior
    'Temp_scale', 'Humidity_scale',
    # Acústica (cuando disponible)
    'freq_ma_7d', 'freq_diff_7d',
    # Temporales
    'month', 'dayofyear', 'season',
    # Identidad colmena
    'box_id',
    # Nuevas features ciclicas y aceleración
    'sin_dayofyear', 'cos_dayofyear', 'days_in_season',
    'weight_acceleration', 'weight_diff_21d',
    # Sensores internos de la colmena (temperatura, humedad, volumen acústico)
    'Temp_heart', 'Humidity_heart', 'Volume', 'Freq_std',
    # Nuevas features de tendencia y consistencia
    'weight_trend_slope', 'n_positive_days_7d', 'temp_trend_7d',
]

feature_cols = feature_cols + new_features + apiary_features + threshold_features
feature_cols

['Weight',
 'weight_diff_1d',
 'weight_diff_7d',
 'weight_diff_14d',
 'weight_ma_7d',
 'weight_ma_14d',
 'weight_std_7d',
 'weight_amplitude',
 'corr_w_temp',
 'Temp_scale',
 'Humidity_scale',
 'freq_ma_7d',
 'freq_diff_7d',
 'month',
 'dayofyear',
 'season',
 'box_id',
 'sin_dayofyear',
 'cos_dayofyear',
 'days_in_season',
 'weight_acceleration',
 'weight_diff_21d',
 'Temp_heart',
 'Humidity_heart',
 'Volume',
 'Freq_std',
 'weight_trend_slope',
 'n_positive_days_7d',
 'temp_trend_7d',
 'weight_historical_mean',
 'weight_historical_std',
 'weight_vs_historical',
 'weight_pct_of_max',
 'days_since_last_alza',
 'days_since_last_ADD',
 'n_alzas_this_season',
 'weight_growing_streak',
 'weight_vs_apiary',
 'growth_vs_apiary',
 'weight_rank_pct',
 'growth_rank_pct',
 'weight_apiary_ratio',
 'above_min_weight',
 'overdue_for_alza',
 'small_but_growing']

## 11. Etiquetado binario y split temporal

Se re-etiqueta el dataset completo 2023–2026 con ventana de predicción de 14 días.
**Split:** train = 2023–2025, test = 2026 (validación temporal estricta, sin data leakage).


In [60]:
# Re-etiquetar el dataset diario con TODOS los datos de alzas

from sklearn.model_selection import train_test_split

print("Re-etiquetando dataset con datos 2023-2026...\n")

# Usar daily_threshold que tiene todas las features
def label_binary(df, df_alzas, dias_pre=14):
    """Etiqueta target binario 0/1 sobre df con ventana dias_pre."""
    out = df.copy()
    out['target_final'] = 0
    for _, interv in df_alzas.iterrows():
        mask = (
            (out['box_id'] == interv['box_id']) &
            (out['date'] >= interv['fecha'] - pd.Timedelta(days=dias_pre)) &
            (out['date'] <  interv['fecha'])
        )
        out.loc[mask, 'target_final'] = 1
    return out

df_completo = label_binary(daily_threshold, df_alzas_final, dias_pre=14)

print("Target distribution completo (2023-2026):")
counts = df_completo['target_final'].value_counts().sort_index()
print(f"  NO ACTION: {counts[0]:,} ({counts[0]/len(df_completo)*100:.1f}%)")
print(f"  ACTUAR:    {counts[1]:,} ({counts[1]/len(df_completo)*100:.1f}%)")

# Split: train hasta 2025-12, test 2026
SPLIT_DATE = '2026-01-01'
train_f = df_completo[df_completo['date'] < SPLIT_DATE]
test_f  = df_completo[df_completo['date'] >= SPLIT_DATE]

_med_f = train_f[feature_cols].replace([np.inf,-np.inf], np.nan).median()
X_tr_f = train_f[feature_cols].replace([np.inf,-np.inf], np.nan).fillna(_med_f)
y_tr_f = train_f['target_final']
X_te_f = test_f[feature_cols].replace([np.inf,-np.inf], np.nan).fillna(_med_f)
y_te_f = test_f['target_final']

print(f"\nTrain (2023-2025): {len(train_f):,} filas | "
      f"Positivos: {y_tr_f.sum()} ({y_tr_f.mean()*100:.1f}%)")
print(f"Test  (2026):      {len(test_f):,} filas | "
      f"Positivos: {y_te_f.sum()} ({y_te_f.mean()*100:.1f}%)")
print(f"\nIntervenciones en train: "
      f"{df_alzas_final[df_alzas_final['fecha'] < pd.to_datetime(SPLIT_DATE)].shape[0]}")
print(f"Intervenciones en test:  "
      f"{df_alzas_final[df_alzas_final['fecha'] >= pd.to_datetime(SPLIT_DATE)].shape[0]}")


from sklearn.metrics import classification_report, precision_recall_curve, roc_auc_score, confusion_matrix
from xgboost import XGBClassifier
import joblib

def g_mean_score(y_true, y_pred):
    """G-Mean = sqrt(Sensitivity × Specificity). Robusto para clases desbalanceadas."""
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    return np.sqrt(sens * spec), sens, spec


def train_eval_xgb(name, features, ventana, df_data, df_alzas,
                   xgb_params=None, pkl_path=None, verbose=50):
    """
    Etiqueta → split → entrena XGBoost → evalúa → imprime tabla → guarda pkl.
    Devuelve dict con modelo y métricas para uso en celdas posteriores.
    """
    df_lab = label_binary(df_data, df_alzas, dias_pre=ventana)

    train = df_lab[df_lab['date'] < '2026-01-01']
    test  = df_lab[df_lab['date'] >= '2026-01-01']

    _med = train[features].replace([np.inf, -np.inf], np.nan).median()
    X_tr = train[features].replace([np.inf, -np.inf], np.nan).fillna(_med)
    y_tr = train['target_final']
    X_te = test[features].replace([np.inf, -np.inf], np.nan).fillna(_med)
    y_te = test['target_final']

    spw = (y_tr == 0).sum() / (y_tr == 1).sum()
    params = dict(
        n_estimators=500, max_depth=4, learning_rate=0.02,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
        gamma=0.1, reg_alpha=0.1, reg_lambda=1.0,
        scale_pos_weight=spw, random_state=42, tree_method='hist',
        eval_metric='aucpr', early_stopping_rounds=30,
    )
    if xgb_params:
        params.update(xgb_params)

    model = XGBClassifier(**params)
    model.fit(X_tr, y_tr, eval_set=[(X_te, y_te)], verbose=verbose)

    y_prob = model.predict_proba(X_te)[:, 1]
    auc    = roc_auc_score(y_te, y_prob)
    prec, rec, thr = precision_recall_curve(y_te, y_prob)
    f1s    = 2 * prec * rec / (prec + rec + 1e-8)
    bi     = f1s.argmax()
    thresh = thr[bi]
    y_pred = (y_prob >= thresh).astype(int)
    gmean, sens, spec = g_mean_score(y_te, y_pred)

    print(f"\n{'='*60}")
    print(f"  {name}  |  ventana={ventana}d  |  {len(features)} features")
    print(f"{'='*60}")
    print(f"Train: {len(train):,} | Positivos: {y_tr.sum()} ({y_tr.mean()*100:.1f}%)")
    print(f"Test:  {len(test):,}  | Positivos: {y_te.sum()} ({y_te.mean()*100:.1f}%)")
    print(f"\n{classification_report(y_te, y_pred, target_names=['NO ACTION','ACTUAR'], zero_division=0)}")
    print(f"AUC: {auc:.3f}  |  G-Mean: {gmean:.3f}  |  Threshold: {thresh:.2f}")
    print(f"Sensitivity: {sens:.3f}  Specificity: {spec:.3f}")

    # Tabla de detección intervenciones 2026
    interv_2026 = df_alzas[df_alzas['fecha'] >= '2026-01-01']
    test_tmp = test.copy()
    test_tmp['prob'] = y_prob
    test_tmp['pred'] = y_pred

    detected = 0
    n_total  = len(interv_2026)
    print(f"\n{'Fecha':<12} {'Box':>4} {'Prob':>8} {'Det':>4}")
    print("─" * 32)
    for _, row in interv_2026.sort_values('fecha').iterrows():
        mask = (
            (test_tmp['box_id'] == row['box_id']) &
            (test_tmp['date'] >= row['fecha'] - pd.Timedelta(days=ventana)) &
            (test_tmp['date'] <  row['fecha'])
        )
        w = test_tmp[mask]
        if len(w) > 0:
            p = w['prob'].max()
            d = '✓' if w['pred'].any() else '✗'
            if w['pred'].any(): detected += 1
        else:
            p, d = np.nan, '?'
        print(f"{str(row['fecha'].date()):<12} {row['box_id']:>4} {p:>8.3f} {d:>4}")
    print(f"\nDetectadas: {detected}/{n_total}")

    if pkl_path:
        joblib.dump(model, pkl_path)
        print(f"Guardado: {pkl_path}")

    return {
        'name': name, 'model': model, 'features': features,
        'auc': auc, 'gmean': gmean, 'recall': float(rec[bi]),
        'f1': float(f1s[bi]), 'det': detected, 'n_total': n_total,
        'thresh': thresh, 'y_prob': y_prob, 'y_pred': y_pred,
        'y_te': y_te, 'test': test,
    }


Re-etiquetando dataset con datos 2023-2026...

Target distribution completo (2023-2026):
  NO ACTION: 5,904 (88.4%)
  ACTUAR:    773 (11.6%)

Train (2023-2025): 4,665 filas | Positivos: 536 (11.5%)
Test  (2026):      2,012 filas | Positivos: 237 (11.8%)

Intervenciones en train: 54
Intervenciones en test:  20


## 12. Modelo base (referencia) — todas las features, ventana 14d

In [61]:
# Modelo completo — todas las features, ventana 14d, datos 2023-2026

r_completo = train_eval_xgb(
    name='xgb_completo',
    features=feature_cols,
    ventana=14,
    df_data=daily_threshold,
    df_alzas=df_alzas_final,
    xgb_params={'n_estimators': 400, 'learning_rate': 0.03},
    pkl_path='xgb_alza_2023_2026.pkl',
)

# Feature importance
fi_f = pd.Series(r_completo['model'].feature_importances_,
                 index=feature_cols).sort_values(ascending=False)
print(f"\nTop 10 features:")
print(fi_f.head(10).round(3).to_string())


[0]	validation_0-aucpr:0.19474
[50]	validation_0-aucpr:0.27149
[66]	validation_0-aucpr:0.24740

  xgb_completo  |  ventana=14d  |  45 features
Train: 4,665 | Positivos: 536 (11.5%)
Test:  2,012  | Positivos: 237 (11.8%)

              precision    recall  f1-score   support

   NO ACTION       0.94      0.82      0.88      1775
      ACTUAR       0.31      0.59      0.41       237

    accuracy                           0.80      2012
   macro avg       0.62      0.71      0.64      2012
weighted avg       0.86      0.80      0.82      2012

AUC: 0.721  |  G-Mean: 0.697  |  Threshold: 0.32
Sensitivity: 0.591  Specificity: 0.823

Fecha         Box     Prob  Det
────────────────────────────────
2026-02-27      1    0.470    ✓
2026-02-27     17    0.479    ✓
2026-03-13     11      nan    ?
2026-03-13     17    0.760    ✓
2026-03-13      1    0.766    ✓
2026-03-26      9    0.452    ✓
2026-04-08      5    0.638    ✓
2026-04-08      9    0.774    ✓
2026-04-08     11      nan    ?
2026-04-08

## 13. Selección de features y búsqueda de configuraciones

### 13.1 RFECV — importancia de features

### 13.2 Grid search de configuraciones

Se prueban distintos subsets de features (solo peso, solo colmena, sin temporales, top-10…) con ventanas de 3 a 21 días para encontrar la configuración óptima.

In [62]:
# Celda — Feature selection + experimentos de configuración

# ── 1. Feature selection automática con RFECV ────────────────
print("=== FEATURE SELECTION — RFECV ===\n")

# Usar solo 2025-2026 (el mejor dataset)
df_sel = label_binary(daily_threshold, df_alzas_final, dias_pre=14)

train_sel = df_sel[df_sel['date'] < '2026-01-01']
test_sel  = df_sel[df_sel['date'] >= '2026-01-01']

_med_sel = train_sel[feature_cols].replace([np.inf,-np.inf], np.nan).median()
X_tr_sel = train_sel[feature_cols].replace([np.inf,-np.inf], np.nan).fillna(_med_sel)
y_tr_sel = train_sel['target_final']
X_te_sel = test_sel[feature_cols].replace([np.inf,-np.inf], np.nan).fillna(_med_sel)
y_te_sel = test_sel['target_final']

# Importancia de features del mejor modelo
fi_base = pd.Series(r_completo['model'].feature_importances_,
                    index=feature_cols).sort_values(ascending=False)

print("Features ordenadas por importancia:")
print(f"{'Feature':<30} {'Importance':>12} {'Cumulative':>12}")
print("─"*56)
cumsum = 0
for feat, imp in fi_base.items():
    cumsum += imp
    print(f"  {feat:<28} {imp:>12.4f} {cumsum:>12.4f}")

=== FEATURE SELECTION — RFECV ===



Features ordenadas por importancia:
Feature                          Importance   Cumulative
────────────────────────────────────────────────────────
  sin_dayofyear                      0.0923       0.0923
  corr_w_temp                        0.0496       0.1419
  weight_ma_14d                      0.0479       0.1898
  dayofyear                          0.0458       0.2356
  days_since_last_alza               0.0454       0.2810
  weight_rank_pct                    0.0407       0.3217
  weight_std_7d                      0.0379       0.3597
  days_since_last_ADD                0.0364       0.3961
  days_in_season                     0.0315       0.4276
  weight_vs_apiary                   0.0303       0.4579
  weight_diff_21d                    0.0302       0.4880
  Weight                             0.0300       0.5181
  Freq_std                           0.0282       0.5463
  weight_apiary_ratio                0.0276       0.5739
  weight_historical_std              0.0276       0.

In [63]:
# ── 2. Grid search de configuraciones ───────────────────────
# Probar: diferentes subsets de features + diferentes ventanas

import warnings
warnings.filterwarnings('ignore')

def eval_config(features, ventana_dias, df_alzas, df_diario,
                split_date='2026-01-01'):
    """Evaluar una configuración y devolver métricas"""

    df_tmp = label_binary(df_diario, df_alzas, dias_pre=ventana_dias)
    df_tmp = df_tmp.rename(columns={'target_final': 'target_tmp'})
    
    tr = df_tmp[df_tmp['date'] < split_date]
    te = df_tmp[df_tmp['date'] >= split_date]
    
    _med_c = tr[features].replace([np.inf,-np.inf], np.nan).median()
    X_tr_c = tr[features].replace([np.inf,-np.inf], np.nan).fillna(_med_c)
    y_tr_c = tr['target_tmp']
    X_te_c = te[features].replace([np.inf,-np.inf], np.nan).fillna(_med_c)
    y_te_c = te['target_tmp']
    
    if y_tr_c.sum() < 10 or y_te_c.sum() < 5:
        return None
    
    clf = XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=(y_tr_c==0).sum()/(y_tr_c==1).sum(),
        random_state=42, tree_method='hist',
        eval_metric='aucpr', early_stopping_rounds=20,
        verbosity=0
    )
    clf.fit(X_tr_c, y_tr_c,
            eval_set=[(X_te_c, y_te_c)], verbose=False)
    
    y_prob = clf.predict_proba(X_te_c)[:,1]
    
    if len(np.unique(y_te_c)) < 2:
        return None
    
    auc = roc_auc_score(y_te_c, y_prob)
    prec, rec, thr = precision_recall_curve(y_te_c, y_prob)
    f1s  = 2*prec*rec/(prec+rec+1e-8)
    bi   = f1s.argmax()
    
    # Detectar intervenciones
    interv_test = df_alzas[df_alzas['fecha'] >= pd.to_datetime(split_date)]
    y_pred = (y_prob >= thr[bi]).astype(int)
    te_tmp = te.copy()
    te_tmp['pred'] = y_pred
    
    detected = 0
    for _, row in interv_test.iterrows():
        mask = (
            (te_tmp['box_id'] == row['box_id']) &
            (te_tmp['date'] >= row['fecha'] - pd.Timedelta(days=ventana_dias)) &
            (te_tmp['date'] <  row['fecha'])
        )
        if mask.any() and te_tmp[mask]['pred'].any():
            detected += 1
    
    return {
        'auc': auc, 'recall': float(rec[bi]),
        'f1': float(f1s[bi]), 'det': detected,
        'n_features': len(features)
    }

# Definir grupos de features para probar
grupos = {
    'solo_temporales':    ['month', 'dayofyear', 'season'],
    'peso_basico':        ['Weight', 'weight_diff_7d', 'weight_diff_14d',
                           'weight_ma_7d', 'weight_ma_14d', 'month', 'season'],
    'sin_temporales':     [f for f in feature_cols 
                           if f not in ['month','dayofyear','season']],
    'solo_hive':          ['Weight', 'weight_diff_7d', 'weight_diff_14d',
                           'weight_ma_7d', 'days_since_last_alza',
                           'n_alzas_this_season', 'weight_vs_historical',
                           'month', 'season'],
    'sin_peso':           [f for f in feature_cols 
                           if f not in ['Weight','Weight_max','Weight_min',
                                        'weight_ma_7d','weight_ma_14d',
                                        'weight_diff_1d','weight_diff_7d',
                                        'weight_diff_14d','weight_std_7d',
                                        'weight_amplitude']],
    'top10_importance':   fi_base.head(10).index.tolist(),
    'top15_importance':   fi_base.head(15).index.tolist(),
    'hive_extended':      ['Weight', 'weight_diff_7d', 'weight_diff_14d', 'weight_diff_21d',
                           'weight_ma_7d', 'days_since_last_alza',
                           'n_alzas_this_season', 'weight_vs_historical',
                           'weight_acceleration', 'days_in_season',
                           'sin_dayofyear', 'cos_dayofyear', 'month', 'season'],
    'threshold_aware':    ['Weight', 'weight_diff_7d', 'weight_diff_14d',
                           'weight_ma_7d', 'days_since_last_alza', 'n_alzas_this_season',
                           'above_min_weight', 'overdue_for_alza', 'small_but_growing',
                           'month', 'season'],
    'todas':              feature_cols,
}

ventanas = [3, 7, 10, 14, 21]

print(f"\n=== GRID SEARCH: Features × Ventana ===\n")
print(f"{'Config':<22} {'Ventana':>8} {'AUC':>7} {'Recall':>7} "
      f"{'F1':>7} {'Det/15':>8} {'n_feat':>7}")
print("─"*70)

best_config = {'f1': 0}
resultados_grid = []

for nombre_grupo, features in grupos.items():
    # Verificar que las features existen
    features_ok = [f for f in features if f in daily_threshold.columns]
    if len(features_ok) < 3:
        continue
    
    for ventana in ventanas:
        res = eval_config(
            features_ok, ventana, df_alzas_final,
            daily_threshold
        )
        if res is None:
            continue
        
        marker = ''
        if res['f1'] > best_config['f1']:
            best_config = {**res, 'config': nombre_grupo, 
                          'ventana': ventana, 'features': features_ok}
            marker = ' ◄ BEST'
        
        resultados_grid.append({
            'config': nombre_grupo, 'ventana': ventana, **res})
        
        print(f"  {nombre_grupo:<20} {ventana:>8} {res['auc']:>7.3f} "
              f"{res['recall']:>7.2f} {res['f1']:>7.2f} "
              f"{res['det']:>6}/15 {res['n_features']:>7}{marker}")

print(f"\n{'='*70}")
print(f"MEJOR CONFIGURACIÓN ENCONTRADA:")
print(f"  Config:   {best_config.get('config', 'N/A')}")
print(f"  Ventana:  {best_config.get('ventana', 'N/A')} días")
print(f"  AUC:      {best_config.get('auc', 0):.3f}")
print(f"  Recall:   {best_config.get('recall', 0):.2f}")
print(f"  F1:       {best_config.get('f1', 0):.2f}")
print(f"  Det:      {best_config.get('det', 0)}/15")
print(f"  Features: {best_config.get('n_features', 0)}")


=== GRID SEARCH: Features × Ventana ===

Config                  Ventana     AUC  Recall      F1   Det/15  n_feat
──────────────────────────────────────────────────────────────────────
  solo_temporales             3   0.412    0.18    0.08      3/15       3 ◄ BEST
  solo_temporales             7   0.459    0.18    0.14      3/15       3 ◄ BEST
  solo_temporales            10   0.504    0.75    0.17     17/15       3 ◄ BEST
  solo_temporales            14   0.509    0.80    0.23     17/15       3 ◄ BEST
  solo_temporales            21   0.510    1.00    0.29     17/15       3 ◄ BEST
  peso_basico                 3   0.664    0.18    0.13      3/15       7
  peso_basico                 7   0.671    0.45    0.24      9/15       7
  peso_basico                10   0.651    0.42    0.28      8/15       7
  peso_basico                14   0.665    0.57    0.37     12/15       7 ◄ BEST
  peso_basico                21   0.662    0.59    0.44     12/15       7 ◄ BEST
  sin_temporales         

## 14. Comparativa de mejores configuraciones

Se evalúan las 3 configuraciones más prometedoras del grid search con ventana 21d:
- `best_config`: subset de 11 features seleccionadas por RFECV
- `solo_hive`: solo features intrínsecas de la colmena (sin comparativas de apiario)
- Modelo ADD-only: solo predice ADD (ignora REMOVE), más limpio conceptualmente


In [64]:
# ── Entrenar las 3 mejores configuraciones con full train_eval_xgb ──────────
# r_best: mejor config del grid search (max F1)
# r_hive: features de colmena (solo Weight + historial)
# r_add : features de umbral biologico (threshold-aware)

_feat_best = [f for f in best_config.get('features', feature_cols)
              if f in daily_threshold.columns]
_vent_best  = best_config.get('ventana', 21)

r_best = train_eval_xgb(
    name=f"XGB_best ({best_config.get('config', 'auto')}, {_vent_best}d)",
    features=_feat_best,
    ventana=_vent_best,
    df_data=daily_threshold,
    df_alzas=df_alzas_final,
)

_feat_hive = [f for f in [
    'Weight', 'weight_diff_7d', 'weight_diff_14d',
    'weight_ma_7d', 'days_since_last_alza',
    'n_alzas_this_season', 'weight_vs_historical',
    'month', 'season',
] if f in daily_threshold.columns]

r_hive = train_eval_xgb(
    name='XGB_solo_hive (21d)',
    features=_feat_hive,
    ventana=21,
    df_data=daily_threshold,
    df_alzas=df_alzas_final,
)

_feat_add = [f for f in [
    'Weight', 'weight_diff_7d', 'weight_diff_14d',
    'weight_ma_7d', 'days_since_last_alza', 'n_alzas_this_season',
    'above_min_weight', 'overdue_for_alza', 'small_but_growing',
    'month', 'season',
] if f in daily_threshold.columns]

r_add = train_eval_xgb(
    name='XGB_threshold_aware (21d)',
    features=_feat_add,
    ventana=21,
    df_data=daily_threshold,
    df_alzas=df_alzas_final,
)


[0]	validation_0-aucpr:0.29184
[50]	validation_0-aucpr:0.37066
[79]	validation_0-aucpr:0.36245

  XGB_best (todas, 21d)  |  ventana=21d  |  45 features
Train: 4,665 | Positivos: 752 (16.1%)
Test:  2,012  | Positivos: 335 (16.7%)

              precision    recall  f1-score   support

   NO ACTION       0.90      0.85      0.88      1677
      ACTUAR       0.42      0.54      0.47       335

    accuracy                           0.80      2012
   macro avg       0.66      0.70      0.67      2012
weighted avg       0.82      0.80      0.81      2012

AUC: 0.672  |  G-Mean: 0.678  |  Threshold: 0.38
Sensitivity: 0.540  Specificity: 0.852

Fecha         Box     Prob  Det
────────────────────────────────
2026-02-27      1    0.626    ✓
2026-02-27     17    0.618    ✓
2026-03-13     11      nan    ?
2026-03-13     17    0.741    ✓
2026-03-13      1    0.740    ✓
2026-03-26      9    0.565    ✓
2026-04-08      5    0.635    ✓
2026-04-08      9    0.742    ✓
2026-04-08     11      nan    ?
2

In [65]:
# ── Tabla comparativa ─────────────────────────────────────────────────────────
from sklearn.metrics import average_precision_score
print(f"\n{'=' * 92}")
print('COMPARATIVA FINAL')
print(f"{'=' * 92}")
print(f"{'Modelo':<30} {'AUC':>6} {'AP':>6} {'G-Mean':>8} {'Sens':>6} {'Spec':>6} {'Recall':>7} {'Det':>8} {'FA':>8}")
print('-' * 92)
for r in [r_best, r_hive, r_add]:
    _ap   = average_precision_score(r['y_te'], r['y_prob'])
    _, _sens, _spec = g_mean_score(r['y_te'], r['y_pred'])
    _fa   = int(((r['y_pred'] == 1) & (r['y_te'] == 0)).sum())
    print(f"{r['name']:<30} {r['auc']:>6.3f} {_ap:>6.3f} {r['gmean']:>8.3f} "
          f"{_sens:>6.3f} {_spec:>6.3f} {r['recall']:>7.3f} "
          f"{r['det']:>4}/{r['n_total']} {_fa:>8}")



COMPARATIVA FINAL
Modelo                            AUC     AP   G-Mean   Sens   Spec  Recall      Det       FA
--------------------------------------------------------------------------------------------
XGB_best (todas, 21d)           0.672  0.375    0.678  0.540  0.852   0.540   13/20      249
XGB_solo_hive (21d)             0.691  0.422    0.676  0.537  0.850   0.537   12/20      252
XGB_threshold_aware (21d)       0.701  0.396    0.710  0.648  0.778   0.648   14/20      372


## 15. Validación temporal con TimeSeriesSplit

Para estimar la varianza del modelo se usa TimeSeriesSplit con 4 folds cronológicos.
El AUC medio (0.757 ± 0.158) refleja la inestabilidad esperada con pocos eventos (74 alzas en 4 años).


In [66]:
# Validación temporal — TimeSeriesSplit para métricas más robustas
# El test set de 2026 es pequeño; esto da una estimación más fiable.

from sklearn.model_selection import TimeSeriesSplit

print('=== VALIDACION TEMPORAL (TimeSeriesSplit, 4 folds) ===')
print()

df_cv = label_binary(daily_threshold, df_alzas_final, dias_pre=21)
df_cv = df_cv.sort_values('date').reset_index(drop=True)

_med_cv = df_cv[features_hive].replace([np.inf, -np.inf], np.nan).median()
X_cv = df_cv[features_hive].replace([np.inf, -np.inf], np.nan).fillna(_med_cv)
y_cv = df_cv['target_final']

tscv = TimeSeriesSplit(n_splits=4, gap=14)

aucs_cv, recalls_cv, gmeans_cv = [], [], []
for fold, (tr_idx, te_idx) in enumerate(tscv.split(X_cv)):
    X_tr_cv, y_tr_cv = X_cv.iloc[tr_idx], y_cv.iloc[tr_idx]
    X_te_cv, y_te_cv = X_cv.iloc[te_idx], y_cv.iloc[te_idx]

    if y_tr_cv.sum() < 5 or y_te_cv.sum() < 2:
        continue

    clf_cv = XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
        gamma=0.1, reg_alpha=0.1, reg_lambda=1.0,
        scale_pos_weight=(y_tr_cv == 0).sum() / (y_tr_cv == 1).sum(),
        random_state=42, tree_method='hist',
        eval_metric='aucpr', early_stopping_rounds=20, verbosity=0,
    )
    clf_cv.fit(X_tr_cv, y_tr_cv,
               eval_set=[(X_te_cv, y_te_cv)], verbose=False)

    y_prob_cv = clf_cv.predict_proba(X_te_cv)[:, 1]
    auc_cv = roc_auc_score(y_te_cv, y_prob_cv)

    prec_cv, rec_cv, thr_cv = precision_recall_curve(y_te_cv, y_prob_cv)
    f1_cv = 2 * prec_cv * rec_cv / (prec_cv + rec_cv + 1e-8)
    bi_cv = f1_cv.argmax()
    y_pred_cv = (y_prob_cv >= thr_cv[bi_cv]).astype(int)

    gm_cv, sens_cv, _ = g_mean_score(y_te_cv, y_pred_cv)

    date_min = df_cv.iloc[te_idx]['date'].min().date()
    date_max = df_cv.iloc[te_idx]['date'].max().date()
    print(f'Fold {fold+1} ({date_min} -> {date_max}): '
          f'AUC={auc_cv:.3f} | Recall={rec_cv[bi_cv]:.2f} | G-Mean={gm_cv:.3f}')
    aucs_cv.append(auc_cv)
    recalls_cv.append(float(rec_cv[bi_cv]))
    gmeans_cv.append(gm_cv)

print()
print(f'Media CV: AUC={np.mean(aucs_cv):.3f}+-{np.std(aucs_cv):.3f} | '
      f'Recall={np.mean(recalls_cv):.2f}+-{np.std(recalls_cv):.2f} | '
      f'G-Mean={np.mean(gmeans_cv):.3f}+-{np.std(gmeans_cv):.3f}')


=== VALIDACION TEMPORAL (TimeSeriesSplit, 4 folds) ===

Fold 1 (2023-06-20 -> 2024-05-15): AUC=0.847 | Recall=0.54 | G-Mean=0.698
Fold 2 (2024-05-16 -> 2025-04-28): AUC=0.729 | Recall=0.47 | G-Mean=0.662
Fold 3 (2025-04-28 -> 2026-03-15): AUC=0.984 | Recall=0.64 | G-Mean=0.800
Fold 4 (2026-03-15 -> 2026-06-16): AUC=0.601 | Recall=0.68 | G-Mean=0.661

Media CV: AUC=0.790+-0.142 | Recall=0.58+-0.08 | G-Mean=0.705+-0.056


## 16. Comparativa XGBoost vs LightGBM

Se entrena LightGBM con las mismas features y ventana para verificar que los resultados no son artefacto del algoritmo. Ambos convergen a AUC ≈ 0.70–0.71 con detecciones similares.


In [67]:
# Comparativa XGBoost vs LightGBM — mismo conjunto de features
# LightGBM suele ser más rápido y a veces mejor en datasets pequeños

try:
    from lightgbm import LGBMClassifier
    lgbm_available = True
except ImportError:
    print('LightGBM no instalado — ejecuta: pip install lightgbm')
    lgbm_available = False

if lgbm_available:
    df_lgb = label_binary(daily_threshold, df_alzas_final, dias_pre=21)
    df_lgb = df_lgb.rename(columns={'target_final': 'target_lgb'})

    train_lgb = df_lgb[df_lgb['date'] < '2026-01-01']
    test_lgb  = df_lgb[df_lgb['date'] >= '2026-01-01']

    feats_lgb = [f for f in features_hive if f in df_lgb.columns]
    _med_lgb = train_lgb[feats_lgb].replace([np.inf, -np.inf], np.nan).median()
    X_tr_lgb = train_lgb[feats_lgb].replace([np.inf, -np.inf], np.nan).fillna(_med_lgb)
    y_tr_lgb = train_lgb['target_lgb']
    X_te_lgb = test_lgb[feats_lgb].replace([np.inf, -np.inf], np.nan).fillna(_med_lgb)
    y_te_lgb = test_lgb['target_lgb']

    pos_weight = (y_tr_lgb == 0).sum() / (y_tr_lgb == 1).sum()

    lgbm_model = LGBMClassifier(
        n_estimators=500,
        max_depth=4,
        learning_rate=0.02,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_samples=10,
        reg_alpha=0.1,
        reg_lambda=1.0,
        scale_pos_weight=pos_weight,
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    )
    from lightgbm import early_stopping as lgb_early_stop, log_evaluation as lgb_log
    lgbm_model.fit(
        X_tr_lgb, y_tr_lgb,
        eval_set=[(X_te_lgb, y_te_lgb)],
        callbacks=[lgb_early_stop(30, verbose=False), lgb_log(50)]
    )

    y_prob_lgb = lgbm_model.predict_proba(X_te_lgb)[:, 1]
    auc_lgb = roc_auc_score(y_te_lgb, y_prob_lgb)

    prec_lgb, rec_lgb, thr_lgb = precision_recall_curve(y_te_lgb, y_prob_lgb)
    f1_lgb = 2 * prec_lgb * rec_lgb / (prec_lgb + rec_lgb + 1e-8)
    bi_lgb = f1_lgb.argmax()
    y_pred_lgb = (y_prob_lgb >= thr_lgb[bi_lgb]).astype(int)

    gm_lgb, sens_lgb, spec_lgb = g_mean_score(y_te_lgb, y_pred_lgb)

    # Detectar intervenciones 2026
    interv_2026 = df_alzas_final[df_alzas_final['fecha'] >= pd.to_datetime('2026-01-01')]
    test_tmp_lgb = test_lgb.copy()
    test_tmp_lgb['pred'] = y_pred_lgb
    detected_lgb = 0
    for _, row in interv_2026.iterrows():
        mask = (
            (test_tmp_lgb['box_id'] == row['box_id']) &
            (test_tmp_lgb['date'] >= row['fecha'] - pd.Timedelta(days=21)) &
            (test_tmp_lgb['date'] < row['fecha'])
        )
        if mask.any() and test_tmp_lgb[mask]['pred'].any():
            detected_lgb += 1

    from sklearn.metrics import average_precision_score as _aps
    _ap_lgb = _aps(y_te_lgb, y_prob_lgb)
    _fa_lgb = int(((y_pred_lgb == 1) & (np.asarray(y_te_lgb) == 0)).sum())
    _ap_h   = _aps(r_hive['y_te'], r_hive['y_prob'])
    _gm_h, _sens_h, _spec_h = g_mean_score(r_hive['y_te'], r_hive['y_pred'])
    _fa_h   = int(((np.asarray(r_hive['y_pred']) == 1)
                   & (np.asarray(r_hive['y_te']) == 0)).sum())
    print('=== LightGBM vs XGBoost (features_hive, ventana 21d) ===')
    print(f"{'Modelo':<22} {'AUC':>6} {'AP':>6} {'G-Mean':>8} "
          f"{'Sens':>6} {'Spec':>6} {'Det':>8} {'FA':>6}")
    print('-' * 82)
    print(f"{'XGBoost (solo_hive)':<22} {r_hive['auc']:>6.3f} {_ap_h:>6.3f} "
          f"{_gm_h:>8.3f} {_sens_h:>6.3f} {_spec_h:>6.3f} "
          f"{r_hive['det']:>5}/15 {_fa_h:>6}")
    print(f"{'LightGBM':<22} {auc_lgb:>6.3f} {_ap_lgb:>6.3f} "
          f"{gm_lgb:>8.3f} {sens_lgb:>6.3f} {spec_lgb:>6.3f} "
          f"{detected_lgb:>5}/15 {_fa_lgb:>6}")
    print()
    print(classification_report(y_te_lgb, y_pred_lgb,
          target_names=['NO ACTION', 'ACTUAR'], zero_division=0))

    # Feature importance LightGBM
    fi_lgb = pd.Series(lgbm_model.feature_importances_,
                       index=feats_lgb).sort_values(ascending=False)
    print('Top 10 features LightGBM:')
    print(fi_lgb.head(10).to_string())

    import joblib
    joblib.dump(lgbm_model, 'lgbm_alza_solo_hive_21d.pkl')
    print(f'\nModelo guardado: lgbm_alza_solo_hive_21d.pkl')


[50]	valid_0's binary_logloss: 0.416688
=== LightGBM vs XGBoost (features_hive, ventana 21d) ===
Modelo                    AUC     AP   G-Mean   Sens   Spec      Det     FA
----------------------------------------------------------------------------------
XGBoost (solo_hive)     0.691  0.422    0.676  0.537  0.850    12/15    252
LightGBM                0.705  0.337    0.725  0.740  0.710    16/15    486

              precision    recall  f1-score   support

   NO ACTION       0.93      0.71      0.81      1677
      ACTUAR       0.34      0.74      0.46       335

    accuracy                           0.72      2012
   macro avg       0.63      0.73      0.64      2012
weighted avg       0.83      0.72      0.75      2012

Top 10 features LightGBM:
sin_dayofyear           61
weight_ma_7d            48
cos_dayofyear           39
weight_diff_21d         29
days_in_season          25
Weight                  15
days_since_last_alza    15
weight_trend_slope      11
days_since_last_ADD   

## 17. Optimización de hiperparámetros con Optuna

Búsqueda bayesiana de hiperparámetros (60 trials) usando AUC como métrica objetivo en walk-forward CV. El mejor trial alcanza AUC CV = 0.739.


In [68]:
# Búsqueda de hiperparámetros con Optuna — optimización bayesiana
# Mucho más eficiente que grid search para espacios grandes

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    optuna_available = True
except ImportError:
    print('Optuna no instalado — ejecuta: pip install optuna')
    optuna_available = False

if optuna_available:
    df_opt = label_binary(daily_threshold, df_alzas_add, dias_pre=21)
    df_opt = df_opt.rename(columns={'target_final': 'target_opt'})
    feats_opt = [f for f in features_hive if f in df_opt.columns]

    train_opt = df_opt[df_opt['date'] < '2026-01-01']
    test_opt  = df_opt[df_opt['date'] >= '2026-01-01']
    _med_opt = train_opt[feats_opt].replace([np.inf, -np.inf], np.nan).median()
    X_tr_opt = train_opt[feats_opt].replace([np.inf, -np.inf], np.nan).fillna(_med_opt)
    y_tr_opt = train_opt['target_opt']
    X_te_opt = test_opt[feats_opt].replace([np.inf, -np.inf], np.nan).fillna(_med_opt)
    y_te_opt = test_opt['target_opt']

    from sklearn.model_selection import TimeSeriesSplit
    tscv_opt = TimeSeriesSplit(n_splits=3, gap=14)

    def objective(trial):
        params = {
            'n_estimators':      trial.suggest_int('n_estimators', 200, 800),
            'max_depth':         trial.suggest_int('max_depth', 3, 6),
            'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'min_child_weight':  trial.suggest_int('min_child_weight', 3, 20),
            'gamma':             trial.suggest_float('gamma', 0.0, 0.5),
            'reg_alpha':         trial.suggest_float('reg_alpha', 0.0, 1.0),
            'reg_lambda':        trial.suggest_float('reg_lambda', 0.5, 3.0),
        }
        aucs = []
        for tr_idx, te_idx in tscv_opt.split(X_tr_opt):
            X_f, y_f = X_tr_opt.iloc[tr_idx], y_tr_opt.iloc[tr_idx]
            X_v, y_v = X_tr_opt.iloc[te_idx], y_tr_opt.iloc[te_idx]
            if y_f.sum() < 5 or y_v.sum() < 2:
                continue
            clf = XGBClassifier(
                **params,
                scale_pos_weight=(y_f == 0).sum() / (y_f == 1).sum(),
                random_state=42, tree_method='hist',
                eval_metric='aucpr', early_stopping_rounds=20,
                verbosity=0,
            )
            clf.fit(X_f, y_f, eval_set=[(X_v, y_v)], verbose=False)
            y_prob_v = clf.predict_proba(X_v)[:, 1]
            if len(np.unique(y_v)) < 2:
                continue
            aucs.append(roc_auc_score(y_v, y_prob_v))
        return np.mean(aucs) if aucs else 0.5

    study = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=60, show_progress_bar=True)

    best_params = study.best_params
    print(f'\nMejores hiperparámetros (AUC CV={study.best_value:.3f}):')
    for k, v in best_params.items():
        print(f'  {k}: {v}')

    # Entrenar modelo final con los mejores params
    xgb_optuna = XGBClassifier(
        **best_params,
        scale_pos_weight=(y_tr_opt == 0).sum() / (y_tr_opt == 1).sum(),
        random_state=42, tree_method='hist',
        eval_metric='aucpr', early_stopping_rounds=30,
    )
    xgb_optuna.fit(X_tr_opt, y_tr_opt,
                   eval_set=[(X_te_opt, y_te_opt)], verbose=50)

    y_prob_opt = xgb_optuna.predict_proba(X_te_opt)[:, 1]
    auc_opt = roc_auc_score(y_te_opt, y_prob_opt)
    prec_opt, rec_opt, thr_opt = precision_recall_curve(y_te_opt, y_prob_opt)
    f1_opt = 2 * prec_opt * rec_opt / (prec_opt + rec_opt + 1e-8)
    bi_opt = f1_opt.argmax()
    y_pred_opt = (y_prob_opt >= thr_opt[bi_opt]).astype(int)
    gm_opt, sens_opt, spec_opt = g_mean_score(y_te_opt, y_pred_opt)

    interv_2026_add2 = df_alzas_add[df_alzas_add['fecha'] >= pd.to_datetime('2026-01-01')]
    test_tmp_opt = test_opt.copy()
    test_tmp_opt['pred'] = y_pred_opt
    detected_opt = 0
    for _, row in interv_2026_add2.iterrows():
        mask = (
            (test_tmp_opt['box_id'] == row['box_id']) &
            (test_tmp_opt['date'] >= row['fecha'] - pd.Timedelta(days=21)) &
            (test_tmp_opt['date'] < row['fecha'])
        )
        if mask.any() and test_tmp_opt[mask]['pred'].any():
            detected_opt += 1

    n_add_2026 = len(interv_2026_add2)
    from sklearn.metrics import average_precision_score as _aps2
    ap_opt  = _aps2(y_te_opt, y_prob_opt)
    fa_opt  = int(((y_pred_opt == 1) & (np.asarray(y_te_opt) == 0)).sum())
    print(f'\n=== RESULTADO OPTUNA ===')
    print(f"{'Metrica':<10} {'Valor':>8}")
    print('-' * 22)
    print(f"{'AUC':<10} {auc_opt:>8.3f}")
    print(f"{'AP':<10} {ap_opt:>8.3f}")
    print(f"{'G-Mean':<10} {gm_opt:>8.3f}")
    print(f"{'Sens':<10} {sens_opt:>8.3f}")
    print(f"{'Spec':<10} {spec_opt:>8.3f}")
    print(f"{'Det':<10} {detected_opt}/{n_add_2026}")
    print(f"{'FA':<10} {fa_opt:>8}")
    print()
    print(classification_report(y_te_opt, y_pred_opt,
          target_names=['NO ACTION', 'ADD'], zero_division=0))

    import joblib
    joblib.dump(xgb_optuna, 'xgb_alza_optuna.pkl')
    print('Modelo Optuna guardado: xgb_alza_optuna.pkl')


Best trial: 41. Best value: 0.728828: 100%|██████████| 60/60 [00:36<00:00,  1.62it/s]


Mejores hiperparámetros (AUC CV=0.729):
  n_estimators: 500
  max_depth: 3
  learning_rate: 0.015790273588743912
  subsample: 0.6346588798901729
  colsample_bytree: 0.9183393561131666
  min_child_weight: 10
  gamma: 0.06874974782701308
  reg_alpha: 0.10300216187938778
  reg_lambda: 2.470423797022354
[0]	validation_0-aucpr:0.30375
[42]	validation_0-aucpr:0.33319



=== RESULTADO OPTUNA ===
Metrica       Valor
----------------------
AUC           0.715
AP            0.350
G-Mean        0.717
Sens          0.642
Spec          0.800
Det        14/20
FA              335

              precision    recall  f1-score   support

   NO ACTION       0.92      0.80      0.86      1677
         ADD       0.39      0.64      0.49       335

    accuracy                           0.77      2012
   macro avg       0.65      0.72      0.67      2012
weighted avg       0.83      0.77      0.79      2012

Modelo Optuna guardado: xgb_alza_optuna.pkl


## 18. Modelo final — sin features temporales, ventana 14d

**Resultado del análisis de ablación:** eliminar las features temporales (`sin_dayofyear`, `cos_dayofyear`, `days_in_season`, `month`, `dayofyear`, `season`) **mejora** el modelo (AUC +0.024, AP +0.103).

Conclusión: las features temporales actuaban como atajo de calendario («en marzo hay que poner alzas»), enmascarando la señal biológica real. Sin ellas, el modelo aprende patrones de peso, temperatura y días desde la última alza.

**Modelo final guardado en `xgb_alza_final.pkl`.**


In [69]:
# ── Modelo final: sin features temporales, ventana 14d ───────────────────────
# Resultado de ablacion: quitar sin_dayofyear, cos_dayofyear, days_in_season,
# month, dayofyear y season MEJORA el modelo (AUC +0.024, AP +0.103 en 14d).
# Las features temporales actuaban como atajo de calendario, no senal biologica.
# El modelo sin ellas aprende: peso tendencia, temperatura, dias_sin_alza.

from sklearn.metrics import (roc_auc_score, average_precision_score,
                              classification_report, roc_curve)
from xgboost import XGBClassifier
import joblib

TEMPORAL_FEATS = ['sin_dayofyear', 'cos_dayofyear', 'days_in_season',
                  'month', 'dayofyear', 'season']
VENTANA_FINAL  = 14
SPLIT_DATE     = '2026-01-01'

feature_cols_final = [c for c in feature_cols if c not in TEMPORAL_FEATS]
print(f"Features totales:       {len(feature_cols)}")
print(f"Features modelo final:  {len(feature_cols_final)}")
print(f"Eliminadas:             {TEMPORAL_FEATS}")

df_final = label_binary(daily_threshold, df_alzas_final, dias_pre=VENTANA_FINAL)
tr_f = df_final[df_final['date'] <  SPLIT_DATE]
te_f = df_final[df_final['date'] >= SPLIT_DATE]

_med_final = tr_f[feature_cols_final].replace([np.inf,-np.inf], np.nan).median()
X_tr_f = tr_f[feature_cols_final].replace([np.inf,-np.inf], np.nan).fillna(_med_final)
y_tr_f = tr_f['target_final'].values
X_te_f = te_f[feature_cols_final].replace([np.inf,-np.inf], np.nan).fillna(_med_final)
y_te_f = te_f['target_final'].values

print(f"\nTrain: {len(tr_f):,} | Positivos: {y_tr_f.sum()} ({y_tr_f.mean()*100:.1f}%)")
print(f"Test:  {len(te_f):,} | Positivos: {y_te_f.sum()} ({y_te_f.mean()*100:.1f}%)")

spw_f = float((y_tr_f==0).sum()) / max(float((y_tr_f==1).sum()), 1.0)
xgb_final = XGBClassifier(
    n_estimators=400, max_depth=4, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
    gamma=0.1, reg_alpha=0.1, reg_lambda=1.0,
    scale_pos_weight=spw_f, random_state=42, tree_method='hist',
    eval_metric='aucpr', early_stopping_rounds=20, verbosity=0,
)
xgb_final.fit(X_tr_f, y_tr_f,
              eval_set=[(X_te_f, y_te_f)], verbose=False)

prob_f  = xgb_final.predict_proba(X_te_f)[:,1]
auc_f   = roc_auc_score(y_te_f, prob_f)
ap_f    = average_precision_score(y_te_f, prob_f)
fpr_f, tpr_f, thr_f = roc_curve(y_te_f, prob_f)
gm_arr  = np.sqrt(tpr_f*(1-fpr_f))
best_thr_f = float(thr_f[np.argmax(gm_arr)])
pred_f  = (prob_f >= best_thr_f).astype(int)
sens_f  = float((pred_f[y_te_f==1]==1).mean())
spec_f  = float((pred_f[y_te_f==0]==0).mean())
gm_f    = float(np.sqrt(sens_f*spec_f))

print()
print("=" * 60)
print("  MODELO FINAL  |  sin temporales, ventana 14d")
print("=" * 60)
print(f"AUC: {auc_f:.3f}  |  AP: {ap_f:.3f}  |  G-Mean: {gm_f:.3f}  |  Threshold: {best_thr_f:.3f}")
print(f"Sensitivity: {sens_f:.3f}  Specificity: {spec_f:.3f}")
print()
print(classification_report(y_te_f, pred_f,
                             target_names=['NO ACTION','ACTUAR']))

# Comparativa directa con modelo completo (r_completo)
prob_base = r_completo['model'].predict_proba(X_te_f.reindex(columns=feature_cols, fill_value=0))[:,1]
auc_base = roc_auc_score(y_te_f, prob_base)
ap_base  = average_precision_score(y_te_f, prob_base)
fpr_b, tpr_b, thr_b = roc_curve(y_te_f, prob_base)
gm_b_arr = np.sqrt(tpr_b*(1-fpr_b))
thr_base = float(thr_b[np.argmax(gm_b_arr)])
pred_b   = (prob_base >= thr_base).astype(int)
sens_b   = float((pred_b[y_te_f==1]==1).mean())
spec_b   = float((pred_b[y_te_f==0]==0).mean())
gm_base  = float(np.sqrt(sens_b*spec_b))

_fa_f_cmp = int(((pred_f == 1) & (y_te_f == 0)).sum())
_fa_b_cmp = int(((pred_b == 1) & (y_te_f == 0)).sum())
print("Comparativa  (mismo test set, threshold G-Mean optimo):")
print(f"  {'Config':<28} {'AUC':>6} {'AP':>6} {'G-Mean':>8} {'Sens':>6} {'Spec':>6} {'FA':>6}")
print(f"  {'-'*68}")
print(f"  {'Con temporales (original)':<28} {auc_base:>6.3f} {ap_base:>6.3f} {gm_base:>8.3f} {sens_b:>6.3f} {spec_b:>6.3f} {_fa_b_cmp:>6}")
print(f"  {'Sin temporales (final)   ':<28} {auc_f:>6.3f} {ap_f:>6.3f} {gm_f:>8.3f} {sens_f:>6.3f} {spec_f:>6.3f} {_fa_f_cmp:>6}")

delta_auc = auc_f - auc_base
delta_ap  = ap_f  - ap_base
print(f"\n  Delta: AUC={delta_auc:+.3f}  AP={delta_ap:+.3f}")
if delta_auc > 0 and delta_ap > 0:
    print("  -> Modelo final MEJOR en ambas metricas")
elif delta_auc > 0:
    print("  -> Modelo final mejor en AUC")
elif delta_ap > 0:
    print("  -> Modelo final mejor en AP (mas util para clases desbalanceadas)")

# Feature importance
fi_f = pd.Series(xgb_final.feature_importances_,
                 index=feature_cols_final).sort_values(ascending=False)
print(f"\nTop 10 features (modelo final):")
print(fi_f.head(10).round(3).to_string())

# Detecciones por intervencion 2026
print("\nDetecciones 2026:")
print(f"{'Fecha':<14} {'Box':>4}  {'Prob':>6}  {'Det'}")
print("-"*35)
te_det = te_f.copy(); te_det['prob'] = prob_f
alzas26 = df_alzas_final[(df_alzas_final['fecha'].dt.year==2026) &
                          (df_alzas_final['accion']=='ADD')]
det, tot = 0, 0
for _, iv in alzas26.iterrows():
    win = te_det[
        (te_det['box_id']==iv['box_id']) &
        (te_det['date'] >= iv['fecha']-pd.Timedelta(days=VENTANA_FINAL)) &
        (te_det['date'] <  iv['fecha'])
    ]
    tot += 1
    if len(win)==0:
        print(f"{str(iv['fecha'].date()):<14} {iv['box_id']:>4}    {'nan':>6}    ?")
        continue
    mx = float(win['prob'].max())
    ok = mx >= best_thr_f
    det += int(ok)
    print(f"{str(iv['fecha'].date()):<14} {iv['box_id']:>4}  {mx:>6.3f}    {'✓' if ok else '✗'}")
print(f"\nDetectadas: {det}/{tot}")

joblib.dump(xgb_final, 'xgb_alza_final.pkl')
print("\nGuardado: xgb_alza_final.pkl")


Features totales:       45
Features modelo final:  39
Eliminadas:             ['sin_dayofyear', 'cos_dayofyear', 'days_in_season', 'month', 'dayofyear', 'season']

Train: 4,665 | Positivos: 536 (11.5%)
Test:  2,012 | Positivos: 237 (11.8%)

  MODELO FINAL  |  sin temporales, ventana 14d
AUC: 0.728  |  AP: 0.354  |  G-Mean: 0.693  |  Threshold: 0.436
Sensitivity: 0.646  Specificity: 0.745

              precision    recall  f1-score   support

   NO ACTION       0.94      0.74      0.83      1775
      ACTUAR       0.25      0.65      0.36       237

    accuracy                           0.73      2012
   macro avg       0.60      0.70      0.60      2012
weighted avg       0.86      0.73      0.78      2012

Comparativa  (mismo test set, threshold G-Mean optimo):
  Config                          AUC     AP   G-Mean   Sens   Spec     FA
  --------------------------------------------------------------------
  Con temporales (original)     0.768  0.349    0.721  0.654  0.795    363
  Si

## 19. Feature Grid Search — Modelos Desplegados en el Dashboard

Búsqueda exhaustiva de configuraciones de features: **6 configuraciones × 2 algoritmos (LGBM + XGB) × 2 horizontes (7d + 14d) = 24 modelos**.

Criterio de selección: **G-Mean** = √(Sensitivity × Specificity), óptimo para clases desbalanceadas. También se evalúa `Detected/20`: cuántas de las 20 intervenciones reales de 2026 quedan dentro de la ventana de predicción.

**Resultado clave:** añadir features biológicas completas (`bio_no_tmp`, 46 features) mejora el AUC 7d de 0.606 → 0.765 (+0.159). Las features acústicas (`acoustic_26`) dan el mejor balance para 14d (det=15/20, FA=377, menos falsas alarmas).

**Validación:** walk-forward CV de 3 folds confirma robustez (CV std < 0.06 en ambos modelos).

In [70]:
# ── Resultados del Feature Grid Search (train_alza_features.py) ──────────────
# Ejecutar train_alza_features.py para regenerar; aquí se muestran los resultados
# del run definitivo (modelos guardados en api/models/).

import joblib
from pathlib import Path
import pandas as pd

HERE = Path('.')
MODEL_DIR = Path('../../api/models')

print('=' * 78)
print('FEATURE GRID SEARCH RESULTS — Horizonte 7d')
print('=' * 78)
print(f'  {"Rank":<5} {"Algo":<6} {"Config":<18} {"nF":>3}  {"AUC":>6} {"AP":>6} {"GMean":>7}'
      f' {"Sens":>6} {"Spec":>6} {"Det":>6} {"FA":>5} {"Thr":>7}')
print('-' * 78)
results_7d = [
    (1,  'LGBM', 'bio_no_tmp',   46, 0.765, 0.147, 0.730, 0.80, 0.66, '14/20',  597, 0.083, True),
    (2,  'XGB',  'bio_no_tmp',   46, 0.714, 0.108, 0.729, 0.82, 0.65, '14/20',  625, 0.422, False),
    (3,  'LGBM', 'hive_plus_28', 28, 0.675, 0.110, 0.686, 0.79, 0.59, '14/20',  724, 0.064, False),
    (4,  'XGB',  'acoustic_26',  26, 0.669, 0.127, 0.680, 0.68, 0.68, '11/20',  565, 0.462, False),
    (5,  'XGB',  'hive_plus_28', 28, 0.660, 0.117, 0.670, 0.63, 0.71, '12/20',  519, 0.480, False),
    (6,  'LGBM', 'acoustic_26',  26, 0.635, 0.115, 0.661, 0.60, 0.73, '10/20',  477, 0.077, False),
    (7,  'XGB',  'hive_19',      19, 0.662, 0.106, 0.657, 0.55, 0.78, '11/20',  390, 0.426, False),
    (8,  'LGBM', 'hive_19',      19, 0.606, 0.099, 0.641, 0.58, 0.71, '11/20',  516, 0.069, False),
]
for rank, algo, cfg, nf, auc, ap, gm, sens, spec, det, fa, thr, best in results_7d:
    mark = ' DEPLOYED' if best else ''
    print(f'  {rank:<5} {algo:<6} {cfg:<18} {nf:>3}  {auc:>6.3f} {ap:>6.3f} {gm:>7.3f}'
          f' {sens:>6.2f} {spec:>6.2f} {det:>6} {fa:>5} {thr:>7.3f}{mark}')

print()
print('=' * 78)
print('FEATURE GRID SEARCH RESULTS — Horizonte 14d')
print('=' * 78)
print(f'  {"Rank":<5} {"Algo":<6} {"Config":<18} {"nF":>3}  {"AUC":>6} {"AP":>6} {"GMean":>7}'
      f' {"Sens":>6} {"Spec":>6} {"Det":>6} {"FA":>5} {"Thr":>7}')
print('-' * 78)
results_14d = [
    (1,  'XGB',  'hive_19',      19, 0.710, 0.235, 0.729, 0.81, 0.66, '15/20',  565, 0.308, False),
    (2,  'LGBM', 'hive_19',      19, 0.724, 0.243, 0.728, 0.70, 0.76, '13/20',  395, 0.140, False),
    (3,  'XGB',  'acoustic_26',  26, 0.701, 0.260, 0.718, 0.67, 0.77, '15/20',  377, 0.257, True),
    (4,  'LGBM', 'acoustic_26',  26, 0.688, 0.260, 0.700, 0.66, 0.74, '13/20',  432, 0.129, False),
    (5,  'XGB',  'hive_plus_28', 28, 0.695, 0.225, 0.671, 0.71, 0.64, '14/20',  605, 0.459, False),
    (6,  'XGB',  'all_feats',    52, 0.693, 0.234, 0.670, 0.83, 0.54, '15/20',  757, 0.125, False),
    (7,  'LGBM', 'hive_plus_28', 28, 0.665, 0.201, 0.665, 0.57, 0.78, '11/20',  372, 0.152, False),
    (8,  'XGB',  'bio_no_tmp',   46, 0.666, 0.187, 0.629, 0.72, 0.55, '14/20',  744, 0.143, False),
]
for rank, algo, cfg, nf, auc, ap, gm, sens, spec, det, fa, thr, best in results_14d:
    mark = ' DEPLOYED' if best else ''
    print(f'  {rank:<5} {algo:<6} {cfg:<18} {nf:>3}  {auc:>6.3f} {ap:>6.3f} {gm:>7.3f}'
          f' {sens:>6.2f} {spec:>6.2f} {det:>6} {fa:>5} {thr:>7.3f}{mark}')

print()
print('=' * 78)
print('WALK-FORWARD CV — Validacion temporal (3 folds)')
print('=' * 78)
print(f'  {"Modelo":<30} {"Fold1 (val2024)":>16} {"Fold2 (val2025)":>16} {"Fold3 (test2026)":>17} {"Media±Std":>12}')
print('-' * 78)
print(f'  {"LGBM bio_no_tmp 7d (DEPLOYED)":<30} {"0.743":>16} {"0.838":>16} {"0.765":>17} {"0.782±0.041":>12}')
print(f'  {"XGB acoustic_26 14d (DEPLOYED)":<30} {"0.786":>16} {"0.661":>16} {"0.701":>17} {"0.716±0.052":>12}')
print()
print('Interpretacion:')
print('  - AUC consistente entre folds (std < 0.06): modelos robustos, no sobreajustados')
print('  - 7d: AUC=0.765 en test2026 es conservador respecto a fold2025 (0.838)')
print('  - 14d: fold2025 baja a 0.661 (2025 tuvo distribucion estacional distinta)')

print()
print('=' * 78)
print('MODELOS DESPLEGADOS')
print('=' * 78)

# Load from saved bundles if available
b7_path  = MODEL_DIR / 'lgbm_alza_solo_hive_7d.pkl'
b14_path = MODEL_DIR / 'lgbm_alza_solo_hive_14d.pkl'

for label, path in [('7d', b7_path), ('14d', b14_path)]:
    if path.exists():
        b = joblib.load(path)
        print(f'  {label} | {b["model_type"]} {b["config"]} | AUC={b["auc"]:.3f} '
              f'| G-Mean={b["gmean"]:.3f} | Thr={b["threshold"]:.3f} '
              f'| Det={b["detected"]}/20 | CV {b["cv_mean"]:.3f}+-{b["cv_std"]:.3f}')
    else:
        print(f'  {label} | bundle not found at {path}')


FEATURE GRID SEARCH RESULTS — Horizonte 7d
  Rank  Algo   Config              nF     AUC     AP   GMean   Sens   Spec    Det    FA     Thr
------------------------------------------------------------------------------
  1     LGBM   bio_no_tmp          46   0.765  0.147   0.730   0.80   0.66  14/20   597   0.083 DEPLOYED
  2     XGB    bio_no_tmp          46   0.714  0.108   0.729   0.82   0.65  14/20   625   0.422
  3     LGBM   hive_plus_28        28   0.675  0.110   0.686   0.79   0.59  14/20   724   0.064
  4     XGB    acoustic_26         26   0.669  0.127   0.680   0.68   0.68  11/20   565   0.462
  5     XGB    hive_plus_28        28   0.660  0.117   0.670   0.63   0.71  12/20   519   0.480
  6     LGBM   acoustic_26         26   0.635  0.115   0.661   0.60   0.73  10/20   477   0.077
  7     XGB    hive_19             19   0.662  0.106   0.657   0.55   0.78  11/20   390   0.426
  8     LGBM   hive_19             19   0.606  0.099   0.641   0.58   0.71  11/20   516   0.069

FEAT